<a href="https://colab.research.google.com/github/palakbhatt1/PersonaPath-Literacy-Speech-Assessment/blob/main/PersonaPath_DL_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PersonaPath - Literacy Coach (Phase 1)
### AI-Powered Pronunciation & Fluency Analyzer
Run each cell in order. The last cell launches the full UI.

In [1]:
# Cell 1 - Install dependencies
!pip install -q librosa cmudict python-Levenshtein gradio transformers torch torchvision torchaudio
print('Packages ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.4 MB/s eta 0:00:00
Packages ready


In [2]:
# Cell 2 - Imports & model loading
import os, string, math, json, re, time
import torch
import librosa
import numpy as np
import cmudict
import Levenshtein
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

print('Loading Wav2Vec2 model ... (first run takes ~60 s)')
model_name = 'facebook/wav2vec2-base-960h'
processor  = Wav2Vec2Processor.from_pretrained(model_name)
model      = Wav2Vec2ForCTC.from_pretrained(model_name)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
cmu        = cmudict.dict()
print(f'Model loaded on {device}')

Loading Wav2Vec2 model ... (first run takes ~60 s)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


In [3]:
# Cell 3 - Core NLP helpers
def clean_word(word):
    return word.lower().translate(str.maketrans('', '', string.punctuation))

def get_phonemes(word):
    w = clean_word(word)
    return [p.rstrip('012') for p in cmu[w][0]] if w in cmu else None

def phoneme_distance(w1, w2):
    p1, p2 = get_phonemes(w1), get_phonemes(w2)
    if p1 is None or p2 is None:
        return None
    return Levenshtein.distance(' '.join(p1), ' '.join(p2))

def get_score(ref_w, spoken_w):
    if ref_w == spoken_w:
        return 'green'
    dist       = phoneme_distance(ref_w, spoken_w)
    if dist is None:
        dist   = Levenshtein.distance(ref_w, spoken_w)
    similarity = Levenshtein.ratio(ref_w, spoken_w)
    if similarity > 0.55:
        return 'grey'
    if dist is not None and dist <= 4:
        return 'grey'
    return 'red'

def normalize_word(word):
    for sfx, trim in [('ing', 3), ('ed', 2), ('ly', 2)]:
        if word.endswith(sfx):
            return word[:-trim]
    return word

def generate_tip(word_scores):
    red_words  = [ws['word'] for ws in word_scores if ws['score'] == 'red']
    grey_words = [ws['word'] for ws in word_scores if ws['score'] == 'grey']
    if not red_words and not grey_words:
        return 'Perfect reading! Every word was spot-on. Amazing work!'
    if red_words:
        focus = red_words[0]
        return f"Great job! Let's try saying '{focus}' again slowly. Break it into syllables. You're doing amazing!"
    focus = grey_words[0]
    return f"Very nice! Try pronouncing '{focus}' one more time - almost perfect! Keep going!"

print('NLP helpers ready')

NLP helpers ready


In [4]:
# Cell 4 - Pronunciation pipeline
def pronunciation_pipeline(audio_path, reference_text):
    speech, sr = librosa.load(audio_path, sr=16000)
    duration_sec = librosa.get_duration(y=speech, sr=sr)
    duration_min = duration_sec / 60

    inputs = processor(speech, return_tensors='pt', sampling_rate=16000).input_values.to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    predicted_ids  = torch.argmax(logits, dim=-1)
    transcription  = processor.decode(predicted_ids[0]).lower()
    spoken_words   = transcription.split()
    ref_words      = [clean_word(w) for w in reference_text.split()]

    editops     = Levenshtein.editops(ref_words, spoken_words)
    word_scores = [{'word': w, 'spoken': w, 'score': 'green'} for w in ref_words]

    def safe_get(lst, idx):
        return lst[idx] if idx < len(lst) else ''

    for op, i, j in editops:
        if op == 'replace':
            ref_w, spoken_w = ref_words[i], safe_get(spoken_words, j)
            if not spoken_w:
                word_scores[i].update(score='red', spoken='')
                continue
            ref_n, sp_n = normalize_word(ref_w), normalize_word(spoken_w)
            if Levenshtein.ratio(ref_n, sp_n) < 0.2:
                word_scores[i].update(score='red', spoken='')
                continue
            word_scores[i].update(score=get_score(ref_n, sp_n), spoken=spoken_w)
        elif op == 'delete':
            ref_w = ref_words[i]
            color = 'grey' if ref_w in {'the','a','and','to','of','in','on'} else 'red'
            word_scores[i].update(score=color, spoken='')

    wpm       = len(spoken_words) / duration_min if duration_min > 0 else 0
    intervals = librosa.effects.split(speech, top_db=25)
    hesitations = sum(
        1 for k in range(1, len(intervals))
        if (intervals[k][0] - intervals[k-1][1]) / sr > 0.5
    )
    tip     = generate_tip(word_scores)
    green   = sum(1 for ws in word_scores if ws['score'] == 'green')
    total   = len(word_scores)
    accuracy = round((green / total) * 100) if total else 0

    return {
        'transcription': transcription,
        'word_scores'  : word_scores,
        'wpm'          : round(wpm, 1),
        'hesitations'  : hesitations,
        'tip'          : tip,
        'accuracy'     : accuracy,
        'duration'     : round(duration_sec, 1),
    }

print('Pipeline ready')

Pipeline ready


In [5]:
# Cell 5 - Gradio UI
import gradio as gr

PASSAGES = {
    'Primary - The Elephant (Beginner)': 'The elephant sat quietly by the river bank',
    'Primary - Sunny Day (Easy)': 'One sunny day I went to the park with my friends. We played games and laughed together.',
    'Intermediate - Breeze (Medium)': 'The cool breeze felt nice and the sky looked very beautiful. After playing we sat under a big tree and talked happily.',
    'Advanced - Full Story (Hard)': 'One sunny evening I went to the park with my friends. We played games ran around and laughed together. The cool breeze felt nice and the sky looked very beautiful. After playing we sat under a big tree and talked happily. It was a wonderful day and I felt very happy.',
}

def build_passage_html(word_scores):
    color_map = {'green': '#16a34a', 'grey': '#d97706', 'red': '#dc2626'}
    parts = []
    for ws in word_scores:
        c = color_map.get(ws['score'], '#374151')
        tooltip = f"You said: {ws['spoken']}" if ws['spoken'] and ws['spoken'] != ws['word'] else ''
        title   = f' title="{tooltip}"' if tooltip else ''
        underline = ' text-decoration:underline dotted;' if tooltip else ''
        style   = f'color:{c}; font-weight:700; cursor:pointer;{underline}'
        parts.append(f'<span style="{style}"{title}>{ws["word"]}</span>')
    return ' '.join(parts)

def build_result_html(result):
    ph  = build_passage_html(result['word_scores'])
    wpm = result['wpm']; hes = result['hesitations']
    acc = result['accuracy']; tip = result['tip']; dur = result['duration']
    wc  = '#16a34a' if wpm >= 60 else ('#d97706' if wpm >= 30 else '#dc2626')
    hc  = '#16a34a' if hes <= 2  else ('#d97706' if hes <= 5  else '#dc2626')
    bc  = '#1d4ed8' if acc >= 80 else ('#d97706' if acc >= 50 else '#dc2626')
    gn  = sum(1 for ws in result['word_scores'] if ws['score'] == 'green')
    gy  = sum(1 for ws in result['word_scores'] if ws['score'] == 'grey')
    rd  = sum(1 for ws in result['word_scores'] if ws['score'] == 'red')
    wa  = 'Up' if wpm >= 60 else 'Low'
    ha  = 'Low' if hes <= 2  else 'High'
    ws2 = 'On track!' if wpm >= 60 else 'Try reading faster'
    hs2 = 'Great flow!' if hes <= 2 else 'Reduce pauses'
    tr  = result['transcription']

    header = (
        '<div style="font-family:sans-serif;background:#f8fafc;border-radius:16px;overflow:hidden;box-shadow:0 4px 24px rgba(0,0,0,0.08);">'
        '<div style="background:linear-gradient(135deg,#1e3a8a,#2563eb);padding:18px 24px;display:flex;align-items:center;gap:12px;">'
        '<div style="background:rgba(255,255,255,0.15);border-radius:50%;width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:22px;">&#127891;</div>'
        '<div><div style="color:#fff;font-size:17px;font-weight:700;">PersonaPath - Literacy Coach</div>'
        '<div style="color:#93c5fd;font-size:12px;">Phase 1 - Reading Analysis</div></div>'
        '<div style="margin-left:auto;background:rgba(255,255,255,0.15);border-radius:8px;padding:6px 14px;color:#fff;font-size:13px;">Active Feedback</div>'
        '</div>'
    )
    grid_open = '<div style="padding:24px;display:grid;grid-template-columns:1fr 1fr;gap:20px;">'

    left = (
        '<div style="display:flex;flex-direction:column;gap:16px;">'
        '<div style="background:#fff;border-radius:12px;padding:20px;border:1px solid #e2e8f0;">'
        '<div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:14px;">'
        '<span style="font-size:14px;font-weight:600;color:#374151;">Reading Passage</span>'
        '<span style="background:#dbeafe;color:#1d4ed8;font-size:10px;font-weight:700;padding:3px 8px;border-radius:20px;">LABEL: PRIMARY</span>'
        '</div>'
        f'<div style="font-size:20px;line-height:1.8;color:#1e293b;font-weight:500;">{ph}</div>'
        '<div style="margin-top:14px;font-size:11px;color:#6b7280;">TRANSFORMER-BASED ALIGNMENT (WAV2VEC 2.0)</div>'
        '</div>'
        '<div style="background:#fff;border-radius:12px;padding:16px;border:1px solid #e2e8f0;display:flex;gap:20px;flex-wrap:wrap;align-items:center;">'
        '<div style="display:flex;align-items:center;gap:6px;"><span style="width:12px;height:12px;border-radius:50%;background:#16a34a;display:inline-block;"></span><span style="font-size:12px;color:#374151;">Correct</span></div>'
        '<div style="display:flex;align-items:center;gap:6px;"><span style="width:12px;height:12px;border-radius:50%;background:#d97706;display:inline-block;"></span><span style="font-size:12px;color:#374151;">Close</span></div>'
        '<div style="display:flex;align-items:center;gap:6px;"><span style="width:12px;height:12px;border-radius:50%;background:#dc2626;display:inline-block;"></span><span style="font-size:12px;color:#374151;">Needs work</span></div>'
        '</div>'
        '<div style="background:#fff;border-radius:12px;padding:16px;border:1px solid #e2e8f0;">'
        '<div style="font-size:13px;font-weight:600;color:#374151;margin-bottom:12px;">Word Breakdown</div>'
        '<div style="display:flex;gap:12px;">'
        f'<div style="flex:1;text-align:center;background:#f0fdf4;border-radius:8px;padding:12px;"><div style="font-size:24px;font-weight:800;color:#16a34a;">{gn}</div><div style="font-size:11px;color:#6b7280;">Correct</div></div>'
        f'<div style="flex:1;text-align:center;background:#fffbeb;border-radius:8px;padding:12px;"><div style="font-size:24px;font-weight:800;color:#d97706;">{gy}</div><div style="font-size:11px;color:#6b7280;">Close</div></div>'
        f'<div style="flex:1;text-align:center;background:#fef2f2;border-radius:8px;padding:12px;"><div style="font-size:24px;font-weight:800;color:#dc2626;">{rd}</div><div style="font-size:11px;color:#6b7280;">Missed</div></div>'
        '</div></div></div>'
    )

    right = (
        '<div style="display:flex;flex-direction:column;gap:16px;">'
        '<div style="background:#fff;border-radius:12px;padding:18px;border:1px solid #e2e8f0;">'
        '<div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">'
        '<div style="background:linear-gradient(135deg,#7c3aed,#4f46e5);border-radius:50%;width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:18px;">&#129302;</div>'
        '<div><div style="font-size:13px;font-weight:700;color:#1e293b;">Coaching Corner</div>'
        '<div style="font-size:11px;color:#16a34a;font-weight:600;">Active Feedback</div></div></div>'
        f'<div style="font-size:14px;color:#374151;line-height:1.6;background:#f8fafc;border-radius:8px;padding:12px;font-style:italic;">"{tip}"</div>'
        '</div>'
        '<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;">'
        f'<div style="background:#fff;border-radius:12px;padding:16px;border:1px solid #e2e8f0;text-align:center;">'
        f'<div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:8px;"><span style="font-size:11px;color:#6b7280;font-weight:600;">WORDS / MIN</span><span style="background:{wc};color:#fff;font-size:10px;font-weight:700;padding:2px 6px;border-radius:10px;">{wa}</span></div>'
        f'<div style="font-size:36px;font-weight:800;color:{wc};">{int(wpm)}</div><div style="font-size:10px;color:#9ca3af;">{ws2}</div></div>'
        f'<div style="background:#fff;border-radius:12px;padding:16px;border:1px solid #e2e8f0;text-align:center;">'
        f'<div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:8px;"><span style="font-size:11px;color:#6b7280;font-weight:600;">HESITATIONS</span><span style="background:{hc};color:#fff;font-size:10px;font-weight:700;padding:2px 6px;border-radius:10px;">{ha}</span></div>'
        f'<div style="font-size:36px;font-weight:800;color:{hc};">{hes}</div><div style="font-size:10px;color:#9ca3af;">{hs2}</div></div>'
        '</div>'
        '<div style="background:#fff;border-radius:12px;padding:16px;border:1px solid #e2e8f0;">'
        '<div style="display:flex;justify-content:space-between;margin-bottom:8px;">'
        f'<span style="font-size:13px;font-weight:600;color:#374151;">Pronunciation Accuracy</span>'
        f'<span style="font-size:13px;font-weight:700;color:{bc};">{acc}%</span></div>'
        f'<div style="background:#e2e8f0;border-radius:999px;height:10px;overflow:hidden;"><div style="background:{bc};width:{acc}%;height:100%;border-radius:999px;"></div></div>'
        f'<div style="margin-top:8px;font-size:11px;color:#6b7280;">Duration: {dur}s</div></div>'
        '<div style="background:#f1f5f9;border-radius:12px;padding:14px;border:1px solid #e2e8f0;">'
        '<div style="font-size:11px;font-weight:700;color:#6b7280;letter-spacing:0.5px;margin-bottom:6px;">WHAT WE HEARD</div>'
        f'<div style="font-size:13px;color:#374151;font-style:italic;">"{tr}"</div>'
        '</div></div>'
    )

    return header + grid_open + left + right + '</div></div>'

def analyze(audio, passage_choice, custom_text, use_custom):
    if audio is None:
        return "<div style='padding:20px;color:#dc2626;font-weight:600;'>Please upload or record audio first.</div>"
    ref_text = custom_text.strip() if use_custom and custom_text.strip() else PASSAGES.get(passage_choice, list(PASSAGES.values())[0])
    try:
        result = pronunciation_pipeline(audio, ref_text)
        return build_result_html(result)
    except Exception as e:
        return f"<div style='padding:20px;color:#dc2626;font-weight:600;'>Error: {str(e)}</div>"

def update_passage_preview(choice):
    return PASSAGES.get(choice, '')

CSS = '''
.gradio-container { background: #f1f5f9 !important; }
.gr-button-primary { background: linear-gradient(135deg,#1d4ed8,#7c3aed) !important; border:none !important; font-weight:700 !important; }
'''

PLACEHOLDER = '''
<div style='background:#fff;border-radius:16px;padding:60px 40px;text-align:center;border:2px dashed #cbd5e1;color:#94a3b8;'>
  <div style='font-size:48px;margin-bottom:16px;'>&#127897;</div>
  <div style='font-size:18px;font-weight:600;color:#475569;'>Ready to analyze</div>
  <div style='font-size:14px;margin-top:8px;'>Select a passage, record audio, then click Analyze</div>
</div>'''

with gr.Blocks(css=CSS, title='PersonaPath - Literacy Coach') as demo:
    gr.HTML("""<div style='text-align:center;padding:24px 0 8px;'>
      <h1 style='font-size:32px;font-weight:800;color:#1e3a8a;letter-spacing:-0.5px;margin:0;'>&#127891; PersonaPath</h1>
      <p style='font-size:14px;color:#64748b;margin:6px 0 0;'>AI-Powered Literacy Coach &nbsp;&#183;&nbsp; Phase 1: Reading &amp; Pronunciation</p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown('### Passage Selection')
            passage_choice = gr.Dropdown(choices=list(PASSAGES.keys()), value=list(PASSAGES.keys())[0], label='Select a passage', interactive=True)
            passage_preview = gr.Textbox(value=list(PASSAGES.values())[0], label='Passage text (read this aloud)', lines=4, interactive=False)
            use_custom = gr.Checkbox(label='Use custom passage instead', value=False)
            custom_text = gr.Textbox(label='Custom passage', placeholder='Type or paste any passage here ...', lines=4, visible=False)
            use_custom.change(lambda v: gr.update(visible=v), use_custom, custom_text)
            passage_choice.change(update_passage_preview, passage_choice, passage_preview)
            gr.Markdown('### Record or Upload Audio')
            audio_input = gr.Audio(sources=['microphone', 'upload'], type='filepath', label='Read the passage aloud')
            analyze_btn = gr.Button('Analyze Pronunciation', variant='primary', size='lg')
            gr.HTML("<div style='font-size:12px;color:#94a3b8;margin-top:8px;'>Tips: Speak clearly at a natural pace &bull; Quiet environment works best &bull; Max 2 minutes</div>")
        with gr.Column(scale=8):
            result_html = gr.HTML(value=PLACEHOLDER)

    analyze_btn.click(fn=analyze, inputs=[audio_input, passage_choice, custom_text, use_custom], outputs=[result_html])
    gr.HTML("<div style='text-align:center;padding:20px;color:#94a3b8;font-size:12px;'>PersonaPath &middot; Phase 1 &middot; Literacy Coach &nbsp;|&nbsp; Powered by Wav2Vec 2.0 + CMU Pronouncing Dictionary</div>")

print('Launching PersonaPath UI ...')
demo.launch(share=True, debug=False)

/tmp/ipykernel_608/1539698059.py:130: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='PersonaPath - Literacy Coach') as demo:


Launching PersonaPath UI ...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7aea219dfad3705f0b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
